Korzystanie z narzędzi generatywnej AI w rozwiązywaniu zadań nie jest dozwolone

<img src="no_AI.png" alt="Use of AI allowed only when properly documented " width="100" height="100">

---
# Klasyfikacja niezbalansowana
---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from ucimlrepo import fetch_ucirepo 
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.under_sampling import EditedNearestNeighbours
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

In [2]:
sklearn.set_config(transform_output="pandas")
sklearn.__version__

'1.7.2'

# Przykład - [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)

W tym zadaniu użyjemy zbioru danych bankowych. Naszym celem jest przewidzenie, czy klient zdecyduje się na subskrypcję lokaty terminowej.

## Załadowanie danych

In [3]:
bank_marketing = fetch_ucirepo(id=222) 

In [4]:
X = bank_marketing.data.features 
y = bank_marketing.data.targets['y'] 

In [8]:
# # metadane
bank_marketing.metadata

{'uci_id': 222,
 'name': 'Bank Marketing',
 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing',
 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv',
 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).',
 'area': 'Business',
 'tasks': ['Classification'],
 'characteristics': ['Multivariate'],
 'num_instances': 45211,
 'num_features': 16,
 'feature_types': ['Categorical', 'Integer'],
 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'],
 'target_col': ['y'],
 'index_col': None,
 'has_missing_values': 'yes',
 'missing_values_symbol': 'NaN',
 'year_of_dataset_creation': 2014,
 'last_updated': 'Fri Aug 18 2023',
 'dataset_doi': '10.24432/C5K306',
 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'],
 'intro_paper': {'ID': 277,
  'type': 'NATIVE',
  'title': 'A data-driven a

In [9]:
# # dane
bank_marketing.variables

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,None,no
1,job,Feature,Categorical,Occupation,"type of job (categorical: 'admin.','blue-colla...",None,no
2,marital,Feature,Categorical,Marital Status,"marital status (categorical: 'divorced','marri...",None,no
3,education,Feature,Categorical,Education Level,"(categorical: 'basic.4y','basic.6y','basic.9y'...",None,no
4,default,Feature,Binary,None,has credit in default?,None,no
5,balance,Feature,Integer,None,average yearly balance,euros,no
6,housing,Feature,Binary,None,has housing loan?,None,no
7,loan,Feature,Binary,None,has personal loan?,None,no
8,contact,Feature,Categorical,None,contact communication type (categorical: 'cell...,None,yes
9,day_of_week,Feature,Date,None,last contact day of the week,None,no


### Wybór cech wyjaśniających

In [10]:
X_sel = X[['balance', 'duration', 'education']]

## Wstępne przetwarzanie danych

### Brakujące dane i stopień niezbalansowania

In [11]:
len(X_sel), len(X_sel.dropna())

(45211, 43354)

Usunięcie danych nie zabiera nam wiele rekordów, ale jak wpływa to na stopień niezbalansowania?

In [12]:
y.value_counts(normalize=True) * 100

y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64

In [13]:
idx = X_sel.dropna().index  # indeksy, które przetrwają
y.loc[idx].value_counts(normalize=True) * 100

y
no     88.381695
yes    11.618305
Name: proportion, dtype: float64

Bardzo nieznacznie... Dokonujemy zatem podstawienia:

In [14]:
X_sel = X_sel.loc[idx]
y_sel = y.loc[idx]

### Kodowanie zmiennej wyjaśnianej

In [15]:
label_encoder = LabelEncoder()

In [16]:
y_trans = label_encoder.fit_transform(y_sel.values)

### Kodowanie zmiennych wyjaśniających

In [17]:
enc = OneHotEncoder(drop='first', sparse_output=False)

In [18]:
X_trans = X_sel.drop(columns=['education']).join(
    enc.fit_transform(X_sel[['education']]))

## Podział na dane treningowe i walidacyjne

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X_trans, y_trans,
                                                    test_size=0.2, random_state=0)

In [20]:
np.unique(y_train, return_counts=True)

(array([0, 1]), array([30625,  4058]))

## Regresja logistyczna

In [21]:
logreg = LogisticRegression(random_state=0).fit(X_train, y_train)

In [22]:
y_train_pred = logreg.predict(X_train)
y_train_pred_proba = logreg.predict_proba(X_train)[:, 1]
y_test_pred = logreg.predict(X_test)
y_test_pred_proba = logreg.predict_proba(X_test)[:, 1]

In [23]:
confusion_matrix(y_train, y_train_pred)

array([[30173,   452],
       [ 3358,   700]])

In [24]:
confusion_matrix(y_test, y_test_pred)

array([[7570,  122],
       [ 825,  154]])

In [25]:
roc_auc_score(y_train, y_train_pred_proba), roc_auc_score(y_test, y_test_pred_proba)

(0.8148638657828828, 0.8148813592993158)

In [26]:
average_precision_score(y_train, y_train_pred_proba), average_precision_score(y_test, y_test_pred_proba)

(0.40083384565216906, 0.3761873528544216)

### ENN

In [27]:
enn = EditedNearestNeighbours() 

In [28]:
X_train_enn, y_train_enn = enn.fit_resample(X_train, y_train)

In [29]:
logreg = LogisticRegression(random_state=0).fit(X_train_enn, y_train_enn)

In [30]:
y_train_pred = logreg.predict(X_train_enn)
y_train_pred_proba = logreg.predict_proba(X_train_enn)[:, 1]
y_test_pred = logreg.predict(X_test)
y_test_pred_proba = logreg.predict_proba(X_test)[:, 1]

In [32]:
confusion_matrix(y_train_enn, y_train_pred)

array([[23239,   458],
       [ 2363,  1695]])

In [33]:
confusion_matrix(y_test, y_test_pred)

array([[7155,  537],
       [ 573,  406]])

In [34]:
roc_auc_score(y_train_enn, y_train_pred_proba), roc_auc_score(y_test, y_test_pred_proba)

(0.8722899524186297, 0.8191967617417669)

In [35]:
average_precision_score(y_train_enn, y_train_pred_proba), average_precision_score(y_test, y_test_pred_proba)

(0.6499557568399669, 0.3782880147517823)

Jest pewien potencjał, choć tylko na zbiorze treningowym